# SCM — Colab Smoke Test (self-contained)

End-to-end **QLoRA plumbing test** on a free Colab **T4**, in one notebook, with
no repo files to upload. It proves the loop works: 4-bit model loads → LoRA
attaches → a few training steps run → the model still generates code.
This is a plumbing test, **not** a real training run.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Rules honored: APPS **train** split only; LiveCodeBench is never touched.

In [1]:
# 0) GPU check -- need compute capability >= (7,0). T4 is (7,5). P100 (6,0) fails.
import torch
print(torch.cuda.get_device_name(0))
print("compute capability:", torch.cuda.get_device_capability(0))

Tesla T4
compute capability: (7, 5)


In [2]:
# 1) Install Unsloth (pulls a compatible torch/transformers/peft/trl set).
%pip install -q unsloth
# If you see a dependency-resolver conflict: Runtime -> Restart session, re-run.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.6/72.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

## 1. Load the base model in 4-bit and attach LoRA adapters

In [3]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",  # pre-quantized
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,        # auto (fp16 on T4)
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 2. Load a small slice of APPS (train split only) and format it

> `datasets` >= 4.0 removed dataset *scripts* and `trust_remote_code`, and
> `codeparrot/apps` is script-based, so we load its **Parquet export** instead
> (same data, no script).

In [4]:
from huggingface_hub import HfApi
from datasets import load_dataset
import json

def load_apps_split(split, n=None):
    """Load codeparrot/apps from its auto-generated Parquet export (no script)."""
    rev = "refs/convert/parquet"
    files = [f for f in HfApi().list_repo_files("codeparrot/apps",
             repo_type="dataset", revision=rev) if f.endswith(".parquet")]
    configs = sorted({f.split("/")[0] for f in files})
    config = "all" if "all" in configs else configs[0]
    sel = [f for f in files if f.startswith(config + "/")
           and (f"/{split}/" in f or f"/{split}-" in f or f.endswith(f"/{split}.parquet"))]
    data_files = [f"hf://datasets/codeparrot/apps@{rev}/{f}" for f in sel]
    ds = load_dataset("parquet", data_files=data_files, split="train")
    return ds.select(range(n)) if n else ds

# TRAIN split only. Never the test split here, never LiveCodeBench.
raw = load_apps_split("train", n=50)
print("loaded", len(raw), "APPS train problems")

def build_prompt(ex):
    io = json.loads(ex["input_output"]) if ex["input_output"] else {}
    io_hint = "Use Call-Based format\n" if isinstance(io, dict) and io.get("fn_name") else "Use Standard Input format\n"
    sc = f"\n{ex['starter_code']}\n" if (ex.get("starter_code") or "").strip() else "\n"
    return f"QUESTION:\n{ex['question']}\n{sc}{io_hint}ANSWER:\n"

def first_solution(ex):
    sols = json.loads(ex["solutions"]) if ex["solutions"] else []
    return sols[0] if sols else None

def to_text(ex):
    msgs = [{"role": "user", "content": build_prompt(ex)},
            {"role": "assistant", "content": first_solution(ex)}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = raw.filter(lambda ex: first_solution(ex) is not None).map(to_text)
print("training rows:", len(ds))
print(ds[0]["text"][:800])

all/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 45.0MB            

all/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

loaded 50 APPS train problems


Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

training rows: 50
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
QUESTION:
Polycarp has $n$ different binary words. A word called binary if it contains only characters '0' and '1'. For example, these words are binary: "0001", "11", "0" and "0011100".

Polycarp wants to offer his set of $n$ binary words to play a game "words". In this game, players name words and each next word (starting from the second) must start with the last character of the previous word. The first word can be any. For example, these sequence of words can be named during the game: "0101", "1", "10", "00", "00001".

Word reversal is the operation of reversing the order of the characters. For example, the word "0111" after the reversal becomes "1110", the word "11010" aft


## 3. Train ~20 steps

On the current TRL, `dataset_text_field` / `max_length` / `packing` go in
`SFTConfig`, not as `SFTTrainer` kwargs.

In [5]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps                = 2,
        max_steps                   = 20,      # smoke test only
        learning_rate               = 2e-4,
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ_LEN,
        packing                     = False,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        report_to                   = "none",
    ),
)

# Optional: compute loss only on the solution (mask the problem statement).
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )
    print("prompt-masking: on")
except Exception as e:
    print("prompt-masking unavailable:", e)

trainer.train()   # want: loss logged for ~20 steps, no OOM

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 2 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.721101
2,0.448926
3,0.781685
4,0.770305
5,0.489842
6,0.725532
7,0.843533
8,0.578136
9,0.898857
10,0.451409


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-20/tokenizer_config.json.


TrainOutput(global_step=20, training_loss=0.6320670947432518, metrics={'train_runtime': 236.0695, 'train_samples_per_second': 0.339, 'train_steps_per_second': 0.085, 'total_flos': 3043361205749760.0, 'train_loss': 0.6320670947432518, 'epoch': 1.5714285714285714})

## 4. Confirm the model still generates code

In [6]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user",
         "content": "Write a Python program that reads an integer n from standard input and prints the nth Fibonacci number."}]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python program that reads an integer n from standard input and prints the nth Fibonacci number.
assistant
```python
def fibonacci(n):
    # Base cases for the first two Fibonacci numbers
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        # Recursive call to calculate the Fibonacci sequence
        return fibonacci(n-1) + fibonacci(n-2)

# Read an integer n from standard input
n = int(input("Enter a positive integer: "))

# Calculate the nth Fibonacci number using the defined function
result = fibonacci(n)

# Print the result
print(f"The {n}th Fibonacci number is: {result}")
```

In this solution, we define a recursive function `fibonacci` that calculates the nth Fibonacci number. The Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the two preceding ones. We handle the base cases where `n` is 0 or 1 directly. For other values of `n`, 

## 5. (Optional) Push this smoke model to the Hugging Face Hub

Only if you want to save it. You need an HF **write** token added as a Colab
secret named `HF_TOKEN` (left sidebar key icon -> Add new secret -> name
`HF_TOKEN`, paste the token, enable notebook access).

In [7]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

REPO = "Shaurya-saini/qwen2.5-coder-7b-apps-qlora-smoketest"
model.push_to_hub_merged(REPO, tokenizer, save_method="merged_16bit",
                         token=os.environ["HF_TOKEN"])
print("pushed:", REPO)

config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Shaurya-saini/qwen2.5-coder-7b-apps-qlora-smoketest/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   3%|2         | 31.9MB / 1.09GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [10:43<00:00, 160.75s/it]


Unsloth: Merge process complete. Saved to `/content/Shaurya-saini/qwen2.5-coder-7b-apps-qlora-smoketest`
pushed: Shaurya-saini/qwen2.5-coder-7b-apps-qlora-smoketest


## Done — what next

- If all cells ran without OOM and step 4 printed plausible Python, the pipeline
  is proven. The **real** training run goes on **Kaggle (T4 x2)** using
  `training/train_qlora.py` — see `setup/02_kaggle_training.md`.
- **Evaluation** happens in a separate, clean notebook — see
  `setup/03_evaluation_environment.md`.
- How to run the repo's `.py` / `.sh` files: `setup/00_running_files.md`.